## SRP228410

**paper:** [PMID: 27413049](https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/) - Reverse Transcription Errors and RNA-DNA Differences at Short Tandem Repeats, 2016

**date, curator:** 2026-04-01, Sara Carsanaro

**notes**
* SRA says for disease: complications of small intestinal diverticulitis. discussed with anne, since there do not exist a lot of samples from this species i will annotate and notate the disease in the comments/condition. sample is from testis so not clear if that would be impacted by small intestinal diverticulitis anyways
* rejected 3 libraries, DNA seq
* there are 2 biological replicates with 3 technical replicates each, although they all have the same SRS and SAMN IDs

### annotation summary

In [23]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,testis,UBERON:0000473,testis,perfect match


In [24]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,31,UBERON:0000113,post-juvenile adult stage,other


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP228410"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 9/9 [00:09<00:00,  1.03s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
5,SRX7093678,SRP228410,Illumina HiSeq 2500,SRS5606342,,,,,,testis,31,,,,M,,,9601,TruSeq RNA Sample 

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['testis']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000473'
library.loc[:,'anatName'] = 'testis'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,,,,testis,31,perfect match,not documented,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,,,,testis,31,perfect match,not documented,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,,,,testis,31,perfect match,not documented,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,,,,testis,31,perfect match,not documented,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,,,,testis,31,perfect match,not documented,,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['31']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'other'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,,,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

   #libraryId       SRSId
0  SRX7093683  SRS5606342
1  SRX7093682  SRS5606342
2  SRX7093681  SRS5606342
3  SRX7093680  SRS5606342
4  SRX7093679  SRS5606342
5  SRX7093678  SRS5606342


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
library.loc[:,'sampleAge_value'] = '31'
library.loc[:,'sampleAge_unit'] = 'year'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,,01/04/2026,"Total RNA was extracted with the RN

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-01'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,SAC,2026-04-01,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,SAC,2026-04-01,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,SAC,2026-04-01,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,SAC,2026-04-01,"Total RNA was extracted with the RNeasy Mini kit protocol (Qiagen). The extracted RNA was divided into two aliquots that were utilized to generate two separate sequencing libraries (libraries 910051a and 910051b) using the TruSeq RNA sample preparation kit (Illumina) with the stranded protocol. Each of the libraries 910051a and 910051b was sequenced three times (replicate1, 2, 3).",,BO51,,,,adult,,TRANSCRIPTOMIC,cDNA
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,31,year,,,,,,complications of small intestinal diverticulitis,,SAC,2026-04-01,"Total RNA was extrac

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 27413049, note that animal died from complications of small intestinal diverticulitis'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01
1,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01
2,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01
3,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01
4,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01
5,SRX7093678,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate1,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from complications of small intestinal diverticulitis",complications of small intestinal diverticulitis,,SAC,2026-04-01


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP228410,Genomic and RNA-Seq raw sequence reads from the Sumatran orangutan testis sample,"Transcript variation has important implications for organismal function in health and disease. Most transcriptome studies focus on assessing variation in gene expression levels and isoform representation. Variation at the level of transcript sequence is caused by RNA editing and transcription errors, and leads to nongenetically encoded transcript variants, or RNA–DNA differences (RDDs). Such variation has been understudied, in part because its detection is obscured by reverse transcription (RT) and sequencing errors. It has only been evaluated for intertranscript base substitution differences. Here, we investigated transcript sequence variation for short tandem repeats (STRs). We developed the first maximum-likelihood estimator (MLE) to infer RT error and RDD rates, taking next generation sequencing error rates into account. Using the MLE, we empirically evaluated RT error and RDD rates for STRs in a large-scale DNA and RNA replicated sequencing experiment conducted in a primate species. The RT error rates increased exponentially with STR length and were biased toward expansions. The RDD rates were approximately 1 order of magnitude lower than the RT error rates. The RT error rates estimated with the MLE from a primate data set were concordant with those estimated with an independent method, barcoded RNA sequencing, from a Caenorhabditis elegans data set. Our results have important implications for medical genomics, as STR allelic variation is associated with >40 diseases. STR nonallelic transcript variation can also contribute to disease phenotype. The MLE and empirical rates presented here can be used to evaluate the probability of disease-associated transcripts arising due to RDD.",SRA,,,,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,,10.1093/molbev/msw139,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP228410,Genomic and RNA-Seq raw sequence reads from the Sumatran orangutan testis sample,"Transcript variation has important implications for organismal function in health and disease. Most transcriptome studies focus on assessing variation in gene expression levels and isoform representation. Variation at the level of transcript sequence is caused by RNA editing and transcription errors, and leads to nongenetically encoded transcript variants, or RNA–DNA differences (RDDs). Such variation has been understudied, in part because its detection is obscured by reverse transcription (RT) and sequencing errors. It has only been evaluated for intertranscript base substitution differences. Here, we investigated transcript sequence variation for short tandem repeats (STRs). We developed the first maximum-likelihood estimator (MLE) to infer RT error and RDD rates, taking next generation sequencing error rates into account. Using the MLE, we empirically evaluated RT error and RDD rates for STRs in a large-scale DNA and RNA replicated sequencing experiment conducted in a primate species. The RT error rates increased exponentially with STR length and were biased toward expansions. The RDD rates were approximately 1 order of magnitude lower than the RT error rates. The RT error rates estimated with the MLE from a primate data set were concordant with those estimated with an independent method, barcoded RNA sequencing, from a Caenorhabditis elegans data set. Our results have important implications for medical genomics, as STR allelic variation is associated with >40 diseases. STR nonallelic transcript variation can also contribute to disease phenotype. The MLE and empirical rates presented here can be used to evaluate the probability of disease-associated transcripts arising due to RDD.",SRA,partial,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,,10.1093/molbev/msw139,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '27413049'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/'
experiment.loc[:,'DOI'] = '10.1093/molbev/msw139'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP228410,Genomic and RNA-Seq raw sequence reads from the Sumatran orangutan testis sample,"Transcript variation has important implications for organismal function in health and disease. Most transcriptome studies focus on assessing variation in gene expression levels and isoform representation. Variation at the level of transcript sequence is caused by RNA editing and transcription errors, and leads to nongenetically encoded transcript variants, or RNA–DNA differences (RDDs). Such variation has been understudied, in part because its detection is obscured by reverse transcription (RT) and sequencing errors. It has only been evaluated for intertranscript base substitution differences. Here, we investigated transcript sequence variation for short tandem repeats (STRs). We developed the first maximum-likelihood estimator (MLE) to infer RT error and RDD rates, taking next generation sequencing error rates into account. Using the MLE, we empirically evaluated RT error and RDD rates for STRs in a large-scale DNA and RNA replicated sequencing experiment conducted in a primate species. The RT error rates increased exponentially with STR length and were biased toward expansions. The RDD rates were approximately 1 order of magnitude lower than the RT error rates. The RT error rates estimated with the MLE from a primate data set were concordant with those estimated with an independent method, barcoded RNA sequencing, from a Caenorhabditis elegans data set. Our results have important implications for medical genomics, as STR allelic variation is associated with >40 diseases. STR nonallelic transcript variation can also contribute to disease phenotype. The MLE and empirical rates presented here can be used to evaluate the probability of disease-associated transcripts arising due to RDD.",SRA,partial,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/,10.1093/molbev/msw139,,


#### comments

In [19]:
experiment.loc[:,'comment'] = 'removed DNA libraries, note that animal died from complications of small intestinal diverticulitis'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP228410,Genomic and RNA-Seq raw sequence reads from the Sumatran orangutan testis sample,"Transcript variation has important implications for organismal function in health and disease. Most transcriptome studies focus on assessing variation in gene expression levels and isoform representation. Variation at the level of transcript sequence is caused by RNA editing and transcription errors, and leads to nongenetically encoded transcript variants, or RNA–DNA differences (RDDs). Such variation has been understudied, in part because its detection is obscured by reverse transcription (RT) and sequencing errors. It has only been evaluated for intertranscript base substitution differences. Here, we investigated transcript sequence variation for short tandem repeats (STRs). We developed the first maximum-likelihood estimator (MLE) to infer RT error and RDD rates, taking next generation sequencing error rates into account. Using the MLE, we empirically evaluated RT error and RDD rates for STRs in a large-scale DNA and RNA replicated sequencing experiment conducted in a primate species. The RT error rates increased exponentially with STR length and were biased toward expansions. The RDD rates were approximately 1 order of magnitude lower than the RT error rates. The RT error rates estimated with the MLE from a primate data set were concordant with those estimated with an independent method, barcoded RNA sequencing, from a Caenorhabditis elegans data set. Our results have important implications for medical genomics, as STR allelic variation is associated with >40 diseases. STR nonallelic transcript variation can also contribute to disease phenotype. The MLE and empirical rates presented here can be used to evaluate the probability of disease-associated transcripts arising due to RDD.",SRA,partial,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/,10.1093/molbev/msw139,,"removed DNA libraries, note that animal died from complications of small intestinal diverticulitis"


#### save complete file

In [20]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [42]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [43]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [44]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [45]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58790,SRX29127128,SRP591139,Illumina NovaSeq X Plus,SRS25332894,UBERON:0000992,ovary,UBERON:0007222,late adult stage,,Ovary,155,perfect match,not documented,other,F,GRZ,WT,105023,Smart-Seq2,full_length,polyA,,,Model organism or animal sample from Nothobran...,SAMN48992597,155,day,,,,,"PMID:39975269, Thirteen tissues (bone, brain, ...",,,ANN,2026-03-31
58791,SRX29127127,SRP591139,Illumina NovaSeq X Plus,SRS25332893,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,,Ovary,52,perfect match,not documented,other,F,GRZ,WT,105023,Smart-Seq2,full_length,polyA,,,Model organism or animal sample from Nothobran...,SAMN48992596,52,day,,,,,"PMID:39975269, Thirteen tissues (bone, brain, ...",,,ANN,2026-03-31
58792,SRX7093683,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate3,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from com...",complications of small intestinal diverticulitis,,SAC,2026-04-01
58793,SRX7093682,SRP228410,Illumina MiSeq,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate3,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from com...",complications of small intestinal diverticulitis,,SAC,2026-04-01
58794,SRX7093681,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate2,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from com...",complications of small intestinal diverticulitis,,SAC,2026-04-01
58795,SRX7093680,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,Ppyab_910051a_RNAseq_replicate2,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from com...",complications of small intestinal diverticulitis,,SAC,2026-04-01
58796,SRX7093679,SRP228410,Illumina HiSeq 2500,SRS5606342,UBERON:0000473,testis,UBERON:0000113,post-juvenile adult stage,,testis,31,perfect match,not documented,other,M,,,9601,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,Ppyab_910051b_RNAseq_replicate1,SAMN13178754,31,year,,,,,"PMID: 27413049, note that animal died from com...",complications of small intestinal diverticulitis,,SAC,2026-04-01


In [46]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1139,SRP430823,Transcriptional profiling of aging tissues fro...,"Transcriptional profiling of aging brain, hear...",SRA,total,Bgee 1K,78,SMARTer Stranded RNA-seq Kit,full_length,,PRJNA952180,37828039,https://pmc.ncbi.nlm.nih.gov/articles/PMC10570...,10.1038/s41597-023-02609-x,,prime adult and late adult ste in fish
1140,SRP591139,Killifish transcriptomic aging atlas,This dataset includes gene expression profilin...,SRA,total,Bgee 1K,682,Smart-Seq2,full_length,,PRJNA1274512,39975269,https://pmc.ncbi.nlm.nih.gov/articles/PMC11838...,10.1101/2025.01.28.635350,,"not sure about the protocol, to recheck"
1141,SRP228410,Genomic and RNA-Seq raw sequence reads from th...,Transcript variation has important implication...,SRA,partial,Bgee 1K,6,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA587121,27413049,https://pmc.ncbi.nlm.nih.gov/articles/PMC5026258/,10.1093/molbev/msw139,,"removed DNA libraries, note that animal died f..."


### add annotations to git

In [47]:
! git pull

Already up to date.


In [48]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [49]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [50]:
! git add $git_experiment_path $git_library_path

In [51]:
! git commit -m $commit_message_exp

[develop cb011a7] adding annotated bulk experiment SRP228410
 2 files changed, 172 insertions(+), 165 deletions(-)


In [52]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.00 KiB | 1023.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   c546892..cb011a7  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push